In [7]:
import os 

In [8]:
%pwd

'd:\\Ml_complete_projects\\Chicken-disease-project\\chicken-disease-classification\\research'

In [9]:
os.chdir("../")

In [10]:
%pwd

'd:\\Ml_complete_projects\\Chicken-disease-project\\chicken-disease-classification'

In [11]:
from dataclasses import dataclass 
from pathlib import Path    

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [12]:
from cnnClassifier.constants import * 
from cnnClassifier.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH , params_filepath=PARAMS_FILE_PATH ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])



    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

In [14]:
! pip install urllib3

In [15]:
import os 
import urllib.request as request 
import zipfile
from cnnClassifier import logger 
from cnnClassifier.utils.common import get_size



In [23]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            #os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  


    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [26]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-28 17:09:37,342: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-28 17:09:37,343: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-28 17:09:37,343: INFO: common: created directory at: artifacts]
[2026-03-28 17:09:37,344: INFO: 3750502409: File already exists of size: ~ 61 KB]


BadZipFile: File is not a zip file